# 01 — Data Preparation

This notebook retrieves and prepares bioactivity data for the human Nav1.7 sodium channel (SCN9A) from ChEMBL.

The goal is to construct a clean and reproducible dataset for subsequent exploratory analysis, machine learning, and SHAP-based interpretation.

### Workflow
1. Retrieve Nav1.7 bioactivity records from ChEMBL
2. Filter and standardize IC50 measurements
3. Validate molecular structures
4. Assess data quality and repeated measurements
5. Calculate pIC50 values
6. Generate molecular descriptors
7. Export the cleaned dataset for downstream analysis

## Setup and Dependencies

In [22]:
# Install required packages
%pip install -q chembl_webresource_client rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 54.8 MB/s eta 0:00:00


In [34]:
import pandas as pd

from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import Descriptors
from google.colab import files

import numpy as np

## 1. Retrieve Nav1.7 Bioactivity Data from ChEMBL

Nav1.7 (SCN9A) is a voltage-gated sodium channel involved in pain signaling and is an important target for analgesic drug discovery.

In this section, the ChEMBL database is queried to identify the human Nav1.7 single-protein target. All bioactivity records associated with this target are then retrieved for subsequent filtering and quality assessment.

## Run Configuration

During development, `TEST_MODE` can be enabled to retrieve only a small number of ChEMBL records. This allows the data-processing pipeline to be tested quickly without repeatedly downloading the complete dataset.

Set `TEST_MODE = False` when generating the final research dataset.

In [4]:
# =========================
# RUN CONFIGURATION
# =========================

TEST_MODE = True       # True = retrieve a small sample; False = retrieve all records
TEST_SIZE = 10

In [24]:
# Connect to target API endpoint
target_api = new_client.target

# Search ChEMBL for Nav1.7 targets
target_query = target_api.search('Nav1.7')
targets_df = pd.DataFrame.from_dict(target_query)

# Filter for human single-protein targets
human_target = targets_df[
    (targets_df['target_type'] == 'SINGLE PROTEIN') &
    (targets_df['organism'] == 'Homo sapiens')
]

# Inspect matching targets before selecting the target ID
display(human_target[['target_chembl_id', 'pref_name', 'organism', 'target_type']])

,target_chembl_id,pref_name,organism,target_type
2,CHEMBL4296,Sodium channel protein type 9 subunit alpha,Homo sapiens,SINGLE PROTEIN


### Select the Human Nav1.7 Target

The filtered target table contains the human single-protein matches returned by the ChEMBL search.

After verifying that the first row (`index 0`) corresponds to the human Nav1.7 sodium channel, its ChEMBL target ID is selected for the subsequent bioactivity query.

Using `.iloc[0]` selects the first row by position, and `['target_chembl_id']` extracts its ChEMBL target identifier.

In [25]:
# Select Nav1.7 target
target_chembl_id = human_target.iloc[0]['target_chembl_id']

print(f"Target ID: {target_chembl_id}")

# Connect to activity API endpoint
activity_api = new_client.activity

# Query activities associated with Nav1.7
activity_query = activity_api.filter(
    target_chembl_id=target_chembl_id
)

# Retrieve either a small test sample or the complete dataset
if TEST_MODE:
    res = activity_query[:TEST_SIZE]
    print(f"TEST MODE: retrieving only {TEST_SIZE} records")
else:
    res = activity_query
    print("FULL MODE: retrieving all available records")

# Convert results to DataFrame
df = pd.DataFrame.from_dict(res)

print(f"Records retrieved: {len(df):,}")

Target ID: CHEMBL4296
TEST MODE: retrieving only 10 records
Records retrieved: 10


In [12]:
# 1. Save the DataFrame to Colab's temporary cloud storage
df.to_csv('chembl_nav17_raw.csv', index=False)

# 2. Trigger the download window to save it to your local computer
files.download('chembl_nav17_raw.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Load Previously Retrieved Data

To avoid repeatedly querying the ChEMBL API during development, a previously downloaded raw Nav1.7 dataset can be uploaded and loaded into the notebook.

This step is optional and should be used when resuming the analysis from a saved dataset. If the data were retrieved directly from ChEMBL in the current session, this step can be skipped.

In [27]:
# Upload a previously saved dataset
uploaded = files.upload()

# Get the uploaded filename
file_name = list(uploaded.keys())[0]

# Load into DataFrame
df = pd.read_csv(file_name)

print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns successfully!")

Saving chembl_nav17_data (1).csv to chembl_nav17_data (1) (1).csv
Loaded 17,017 rows and 47 columns successfully!


/tmp/ipykernel_1338/1855308669.py:8: DtypeWarning: Columns (1,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_name)


## 2. Filter for Comparable IC50 Measurements

The raw ChEMBL dataset contains multiple activity types, units, and measurement relations. To create a more consistent dataset for quantitative analysis, records are restricted to:

- IC50 measurements
- nanomolar (nM) standardized units
- exact measurements (`standard_relation = "="`)
- non-missing activity values
- non-missing canonical SMILES

These criteria ensure that the retained records contain directly comparable quantitative IC50 measurements and molecular structures suitable for subsequent analysis.

In [28]:
# Filter for exact IC50 measurements reported in nM
processed_df = df[
    (df['standard_type'] == 'IC50') &
    (df['standard_units'] == 'nM') &
    (df['standard_relation'] == '=') &
    df['standard_value'].notna() &
    df['canonical_smiles'].notna()
].copy()

# Convert IC50 values to numeric
processed_df['standard_value'] = pd.to_numeric(
    processed_df['standard_value'],
    errors='coerce'
)

# Remove nonnumeric, zero, or negative IC50 values
processed_df = processed_df[
    processed_df['standard_value'].notna() &
    (processed_df['standard_value'] > 0)
].copy()

In [21]:
len(processed_df)

9715

## 3. Molecular Structure Cleaning

Before molecular descriptors are calculated, the retained compounds are further standardized based on their molecular structures.

The following criteria are applied:

- Remove multi-component structures represented by disconnected SMILES fragments (`.`), such as salts and mixtures.
- Validate each SMILES string using RDKit.
- Calculate exact molecular weight from the RDKit-parsed structure.
- Retain compounds with molecular weights between 100 and 1000 Da.

These filters reduce structural ambiguity and exclude unusually small or large molecules that may fall outside the intended chemical space for this analysis.

In [36]:
print(f"Starting structure cleanup with {len(processed_df):,} records.")

# 1. Remove multi-component structures
single_component_df = processed_df[
    ~processed_df['canonical_smiles'].str.contains(r'\.', na=False)
].copy()

print(
    f"After removing multi-component structures: "
    f"{len(single_component_df):,} records"
)

# 2. Calculate exact molecular weight and validate SMILES
def get_molecular_weight(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return Descriptors.ExactMolWt(mol)


single_component_df['molecular_weight'] = (
    single_component_df['canonical_smiles']
    .apply(get_molecular_weight)
)

# 3. Remove structures RDKit could not parse
valid_mols_df = single_component_df.dropna(
    subset=['molecular_weight']
).copy()

print(
    f"After RDKit structure validation: "
    f"{len(valid_mols_df):,} records"
)


# 4. Apply molecular-weight range
structure_filtered_df = valid_mols_df[
    valid_mols_df['molecular_weight'].between(100, 1000)
].copy()

print(
    f"After MW filter (100–1000 Da): "
    f"{len(structure_filtered_df):,} records"
)


Starting structure cleanup with 9,715 records.
After removing multi-component structures: 9,690 records
After RDKit structure validation: 9,690 records
After MW filter (100–1000 Da): 9,353 records


## 4. Consolidate Repeated IC50 Measurements and Assess Data Quality

ChEMBL may contain multiple IC50 measurements for the same molecular structure. Records are therefore grouped by canonical SMILES, and the median IC50 is calculated as the representative activity value for each structure.

The median is used because it is less sensitive to extreme measurements than the arithmetic mean.

### Data Quality Classification

Measurement consistency is assessed using the ratio between the maximum and minimum reported IC50 values for each molecular structure:

- **GOOD:** max/min ratio < 3
- **WARNING:** max/min ratio between 3 and 10
- **BAD:** max/min ratio > 10

For a compound with only one available measurement, the maximum and minimum IC50 values are identical, resulting in a ratio of 1. Therefore, single-measurement compounds are classified as `GOOD` under this consistency criterion.

### Confidence Level

Because a single measurement cannot provide the same level of experimental support as repeated measurements, an additional confidence level is assigned to compounds classified as `GOOD`:

- **HIGH:** 3 or more measurements
- **MEDIUM:** 2 measurements
- **LOW:** 1 measurement

This separates measurement consistency from the amount of supporting experimental evidence. The resulting classifications are retained as part of the prepared dataset. Their distributions and chemical characteristics are examined separately in `02_exploratory_analysis.ipynb`.

In [38]:
# Calculate the max/min ratio as a measure of measurement consistency
grouped['max_min_ratio'] = (
    grouped['max_val'] / grouped['min_val']
)


# Classify data quality based on the max/min ratio
conditions = [
    (grouped['max_min_ratio'] < 3),
    (grouped['max_min_ratio'] >= 3) &
    (grouped['max_min_ratio'] <= 10),
    (grouped['max_min_ratio'] > 10)
]

choices = [
    'GOOD',
    'WARNING',
    'BAD'
]

grouped['data_quality'] = np.select(
    conditions,
    choices,
    default='UNCLASSIFIED'
)


# Assign confidence levels to GOOD measurements based on measurement count
conf_conditions = [
    (grouped['data_quality'] == 'GOOD') &
    (grouped['measurement_count'] >= 3),

    (grouped['data_quality'] == 'GOOD') &
    (grouped['measurement_count'] == 2),

    (grouped['data_quality'] == 'GOOD') &
    (grouped['measurement_count'] == 1)
]

conf_choices = [
    'HIGH',
    'MEDIUM',
    'LOW'
]

grouped['confidence_level'] = np.select(
    conf_conditions,
    conf_choices,
    default=''
)


# Validate the resulting classifications
print("--- Data Quality Classification Breakdown ---")
print(grouped['data_quality'].value_counts())

print("\n--- Confidence Level Breakdown for GOOD Molecules ---")
print(
    grouped.loc[
        grouped['data_quality'] == 'GOOD',
        'confidence_level'
    ].value_counts()
)


# Preview the prepared compound-level table
display(grouped.head())


--- Data Quality Classification Breakdown ---
data_quality
GOOD       6738
BAD         373
WARNING     364
Name: count, dtype: int64

--- Confidence Level Breakdown for GOOD Molecules ---
confidence_level
LOW       5986
MEDIUM     706
HIGH        46
Name: count, dtype: int64


,canonical_smiles,median,min_val,max_val,measurement_count,max_min_ratio,data_quality,confidence_level
0,C#Cc1cccc(OC2CN(c3ncnc(Nc4cccc(C(=O)NC)c4)n3)C...,990.0,990.0,990.0,1,1.000000,GOOD,LOW
1,C/C=C/c1cc(C(=O)NS(=O)(=O)N2CCC2)c(F)cc1OCC12C...,6.0,6.0,6.0,1,1.000000,GOOD,LOW
2,C=C(C)[C@@H]1CCC(C)=C[C@H]1c1c(O)cc(CCCCC)cc1O,1820.0,1820.0,1820.0,1,1.000000,GOOD,LOW
3,C=C(C)c1ccccc1N1CCOc2cc(S(=O)(=O)Nc3nccs3)ccc21,2420.0,2420.0,2420.0,1,1.000000,GOOD,LOW
4,C=C(C)c1cn2c(NS(C)(=O)=O)nnc2cc1OCC12CC3CC(CC(...,51.1,51.0,51.2,2,1.003922,GOOD,MEDIUM
